In [1]:
# Step 1 — Load and combine

import pandas as pd

true_df = pd.read_csv("True.csv")
fake_df = pd.read_csv("Fake.csv")
true_df["label"] = "real"
fake_df["label"] = "fake"
isot = pd.concat([true_df, fake_df], ignore_index=True)

print(isot.shape)
print(isot.head())
print(isot["label"].value_counts())

(44898, 5)
                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept transgender recruits o...   
2  Senior U.S. Republican senator: 'Let Mr. Muell...   
3  FBI Russia probe helped by Australian diplomat...   
4  Trump wants Postal Service to charge 'much mor...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2  WASHINGTON (Reuters) - The special counsel inv...  politicsNews   
3  WASHINGTON (Reuters) - Trump campaign adviser ...  politicsNews   
4  SEATTLE/WASHINGTON (Reuters) - President Donal...  politicsNews   

                 date label  
0  December 31, 2017   real  
1  December 29, 2017   real  
2  December 31, 2017   real  
3  December 30, 2017   real  
4  December 29, 2017   real  
label
fake    23481
real    21417
Name: count, dtyp

In [2]:
# Step 2 — Audit before you clean

isot.isnull().sum()
isot.duplicated().sum()
isot["text"].str.len().describe()
isot.groupby("label")["text"].apply(lambda x: x.str.len().mean())
isot["subject"].value_counts()

subject
politicsNews       11272
worldnews          10145
News                9050
politics            6841
left-news           4459
Government News     1570
US_News              783
Middle-east          778
Name: count, dtype: int64

In [3]:
# Step 3 — Handle the Reuters dateline

import re

def strip_dateline(text):
    if pd.isna(text):
        return text
    # Remove patterns like "WASHINGTON (Reuters) - " or "LONDON/PARIS (Reuters) -"
    return re.sub(r"^[A-Z][A-Z\s/,\.\-]+\(Reuters\)\s*[-–—]\s*", "", text)

isot["text_clean"] = isot["text"].apply(strip_dateline)

In [4]:
# Step 4 — Drop empties and duplicates

# Drop rows with missing or trivially short text
isot = isot[isot["text_clean"].notna()]
isot = isot[isot["text_clean"].str.len() > 50]

# Drop exact duplicates of the article text
isot = isot.drop_duplicates(subset=["text_clean"])

# Also worth checking title duplicates
isot = isot.drop_duplicates(subset=["title"])

isot = isot.reset_index(drop=True)

In [5]:
# Step 5 — Normalise text for analysis (lightly)

def normalise_for_model(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)  # neutralise URLs
    text = re.sub(r"\s+", " ", text).strip()           # collapse whitespace
    return text

isot["text_model"] = isot["text_clean"].apply(normalise_for_model)

In [6]:
# Step 6 — Parse the date column

isot["date_parsed"] = pd.to_datetime(isot["date"], errors="coerce")
print(isot.groupby("label")["date_parsed"].agg(["min", "max"]))

             min        max
label                      
fake         NaT        NaT
real  2016-01-13 2017-12-31


In [7]:
# Step 7 — Sanity check the result

print(isot.shape)
print(isot["label"].value_counts())
print(isot["text_clean"].str.len().groupby(isot["label"]).describe())
print(isot.sample(3)[["label", "text_clean"]])

(38126, 8)
label
real    20820
fake    17306
Name: count, dtype: int64
         count         mean          std    min     25%     50%      75%  \
label                                                                      
fake   17306.0  2569.909396  2196.416134   51.0  1675.0  2242.0  3015.75   
real   20820.0  2349.934678  1667.454288  129.0   899.0  2194.5  3188.25   

           max  
label           
fake   51794.0  
real   29781.0  
      label                                         text_clean
13672  real  A 17-year-old Danish girl who offered to fight...
4376   real  U.S. President Donald Trump said on Thursday h...
9507   real  GOP leaders have unleashed a stunning level of...


In [9]:
isot.to_csv("isot_clean.csv", index=False)